In [ ]:
import re
import random
import time
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 10

# To reset seeds later for reproduceability
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
# Use gpu instead of cpu if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# load Ag News dataset
dataset = load_dataset("sh0416/ag_news")

df_train_full = pd.DataFrame(dataset["train"])
df_test = pd.DataFrame(dataset["test"])

#Create train test and dev datasets
df_train, df_dev = train_test_split(
    df_train_full, test_size=0.1, random_state=SEED,
    stratify=df_train_full["label"],
)
df_train = df_train.reset_index(drop=True)
df_dev = df_dev.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Subsample for speed
df_train = df_train.sample(n=5000, random_state=SEED).reset_index(drop=True)
df_dev = df_dev.sample(n=1000,  random_state=SEED).reset_index(drop=True)
df_test = df_test.sample(n=1000, random_state=SEED).reset_index(drop=True)

# Put title and description together
for df in [df_train, df_dev, df_test]:
    df["text"] = df["title"] + " " + df["description"]

# Dataset categories
AG_LABELS = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
NUM_CLASSES = 4


c:\Users\erik\OneDrive\EStuff\University\yr2\nlp\assingment 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Regex to group by words with letters & numbers together, remove punctuation
TOKEN_RE = re.compile(r"[A-Za-z0-9']+")

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

# Tokens for Pads & Unknown
PAD, UNK = "<pad>", "<unk>"

# Get rid of rarely used words
def build_vocab(texts, min_freq: int = 2, max_size: int = 30000) -> dict:
    """
    Build a vocabulary mapping from tokens to integer indices.
    The vocabulary will include only tokens that appear at least `min_freq` times,
    and will be limited to `max_size` tokens (including PAD and UNK).
    """
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    # Reserve 0 for PAD and 1 for UNK.
    vocab = {PAD: 0, UNK: 1}
    for word, freq in counter.most_common():
        if freq < min_freq or len(vocab) >= max_size:
            break
        vocab[word] = len(vocab)
    return vocab

def numericalize(tokens: list, vocab: dict) -> None:
    return [vocab.get(t, vocab[UNK]) for t in tokens]

vocab = build_vocab(df_train["text"])
PAD_IDX = vocab[PAD]

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 64

@dataclass
class Batch:
    x: torch.Tensor # (B, T) token ids
    lengths: torch.Tensor # (B,) true lengths
    y: torch.Tensor  # (B,) labels

# Dataset wrapper for vocab
class TextDataset(Dataset):
    def __init__(self, df, vocab: dict, max_len: int=MAX_LEN):
        self.df, self.vocab, self.max_len = df, vocab, max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        # Convert row -> token -> number
        ids = numericalize(tokenize(str(row["text"])), self.vocab)[: self.max_len]
        return ids or [self.vocab[UNK]], int(row["label"]) - 1

# 
def collate(batch: list) -> Batch:
    """Collate function to convert a list of samples into a batch."""
    # batch: list of (ids_list, label)
    # lengths needed for LSTM
    lengths = torch.tensor([len(x) for x, _ in batch], dtype=torch.long)
    # padd with <pad>
    x = torch.full((len(batch), int(lengths.max())), PAD_IDX, dtype=torch.long)
    y = torch.tensor([lbl for _, lbl in batch], dtype=torch.long)
    for i, (ids, _) in enumerate(batch):
        x[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)
    return Batch(x=x, lengths=lengths, y=y)

# Batch, shuffle + load
train_loader = DataLoader(TextDataset(df_train, vocab), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
dev_loader = DataLoader(TextDataset(df_dev,   vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader = DataLoader(TextDataset(df_test,  vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

